**PROMPT**
Act as a senior Python backend developer. Build a complete, production-quality REST API using Flask that satisfies the following requirements:

Endpoints:

GET /square/<int:number> — Accepts an integer via the URL path and returns its square as JSON.
POST /sum — Accepts a JSON request body containing a key "numbers" (a list of integers/floats) and returns their sum as JSON.

Requirements:

Use Flask with clean, modular, well-commented code.
Validate all inputs. Never let the server crash or return an unhandled stack trace — return structured JSON error messages with correct HTTP status codes (400 for bad input, 404 for invalid routes, 500 for unexpected errors) instead.
Handle edge cases: missing JSON body, missing "numbers" key, empty list, non-numeric values, malformed JSON.
Include a health-check root endpoint (GET /).
Make the code runnable both as a standalone app.py script (via flask run / python app.py) and inside a Google Colab notebook (where app.run() blocks the notebook — use a background-thread-safe server instead, and make it safely re-runnable without "Address already in use" errors).
After defining the API, write test code (using the requests library) that calls both endpoints with valid and invalid inputs, and prints the results, to prove correctness.
Briefly explain, in plain English, how each endpoint works and why the error-handling choices were made.

Deliver the final code in clearly separated, ready-to-run blocks, plus a short explanation of the design decisions.

In [ ]:
!pip install flask flask-cors -q

In [ ]:
from flask import Flask, jsonify, request
from werkzeug.exceptions import HTTPException

app = Flask(__name__)

# ---- Global error handlers so nothing takes the server down ----

@app.errorhandler(404)
def not_found(e):
    return jsonify({"error": "Not found", "message": str(e)}), 404

@app.errorhandler(400)
def bad_request(e):
    return jsonify({"error": "Bad request", "message": str(e)}), 400

@app.errorhandler(HTTPException)
def handle_http_exception(e):
    return jsonify({"error": e.name, "message": e.description}), e.code

@app.errorhandler(Exception)
def handle_unexpected_error(e):
    # Catch-all so an unexpected bug returns JSON instead of crashing the server
    return jsonify({"error": "Internal server error", "message": str(e)}), 500


# ---- Endpoint 1: GET /square/<int:number> ----

@app.route("/square/<int:number>", methods=["GET"])
def square(number):
    result = number ** 2
    return jsonify({"number": number, "square": result}), 200


# ---- Endpoint 2: POST /sum ----

@app.route("/sum", methods=["POST"])
def sum_numbers():
    data = request.get_json(silent=True)  # silent=True avoids raising on bad JSON

    if data is None:
        return jsonify({"error": "Invalid or missing JSON body"}), 400

    if "numbers" not in data:
        return jsonify({"error": "Request must include a 'numbers' key"}), 400

    numbers = data["numbers"]

    if not isinstance(numbers, list) or len(numbers) == 0:
        return jsonify({"error": "'numbers' must be a non-empty list"}), 400

    if not all(isinstance(n, (int, float)) for n in numbers):
        return jsonify({"error": "All items in 'numbers' must be int or float"}), 400

    total = sum(numbers)
    return jsonify({"numbers": numbers, "sum": total}), 200


# ---- Optional: health check ----
@app.route("/", methods=["GET"])
def health():
    return jsonify({"status": "API is running"}), 200

print("Flask app defined successfully.")

Flask app defined successfully.


In [ ]:
import threading
import socket
from werkzeug.serving import make_server

class ServerThread(threading.Thread):
    def __init__(self, app, port=5000):
        threading.Thread.__init__(self)
        self.srv = make_server("127.0.0.1", port, app)
        self.port = port
        self.ctx = app.app_context()
        self.ctx.push()

    def run(self):
        print(f"Server starting on port {self.port}...")
        self.srv.serve_forever()

    def shutdown(self):
        self.srv.shutdown()


def is_port_free(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", port)) != 0


def start_server(app, port=5000):
    global server_thread

    # If a server from a previous run is still alive, shut it down first
    if "server_thread" in globals() and server_thread.is_alive():
        print("Shutting down existing server...")
        server_thread.shutdown()
        server_thread.join()

    # If the port is still taken (e.g. by something else), find a free one
    original_port = port
    while not is_port_free(port):
        port += 1
    if port != original_port:
        print(f"Port {original_port} was busy, using port {port} instead.")

    server_thread = ServerThread(app, port=port)
    server_thread.start()

    import time
    time.sleep(1)
    print(f"Server is running at http://127.0.0.1:{port}")
    return port


PORT = start_server(app, port=5000)

Port 5000 was busy, using port 5001 instead.
Server starting on port 5001...
Server is running at http://127.0.0.1:5001


In [ ]:
import requests

BASE_URL = f"http://127.0.0.1:{PORT}"

resp = requests.get(f"{BASE_URL}/square/5")
print("GET /square/5 ->", resp.status_code, resp.json())

resp = requests.post(f"{BASE_URL}/sum", json={"numbers": [1, 2, 3, 4.5]})
print("POST /sum (valid) ->", resp.status_code, resp.json())

resp = requests.post(f"{BASE_URL}/sum", json={"numbers": ["a", "b"]})
print("POST /sum (invalid types) ->", resp.status_code, resp.json())

resp = requests.post(f"{BASE_URL}/sum", json={})
print("POST /sum (missing key) ->", resp.status_code, resp.json())

resp = requests.get(f"{BASE_URL}/nonexistent")
print("GET /nonexistent ->", resp.status_code, resp.json())

INFO:werkzeug:127.0.0.1 - - [01/Sep/2026 18:09:57] "GET /square/5 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [01/Sep/2026 18:09:57] "POST /sum HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [01/Sep/2026 18:09:57] "POST /sum HTTP/1.1" 400 -
INFO:werkzeug:127.0.0.1 - - [01/Sep/2026 18:09:57] "POST /sum HTTP/1.1" 400 -
INFO:werkzeug:127.0.0.1 - - [01/Sep/2026 18:09:57] "GET /nonexistent HTTP/1.1" 404 -


GET /square/5 -> 200 {'number': 5, 'square': 25}
POST /sum (valid) -> 200 {'numbers': [1, 2, 3, 4.5], 'sum': 10.5}
POST /sum (invalid types) -> 400 {'error': "All items in 'numbers' must be int or float"}
POST /sum (missing key) -> 400 {'error': "Request must include a 'numbers' key"}
GET /nonexistent -> 404 {'error': 'Not found', 'message': '404 Not Found: The requested URL was not found on the server. If you entered the URL manually please check your spelling and try again.'}


In [ ]:
from google.colab import files
files.download('app.py')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>